# Redes Neuronales 2026

---
# ACTIVIDAD EXTRA OPCIONAL (DE ALEX):
1. EXTRA) Podemos hacer esto o lo de la guia anterior usando modelo de neurona de H&H?? Y si usamos el de Izhi?? Implementar.

# Guía de ejercicios N.º 11: Clasificación de imágenes del conjunto Fashion-MNIST con PyTorch

Esta guía continúa con el hilo conceptual utilizado en la guía de *Autoencoder*, pero ahora la tarea es **clasificación supervisada**.

En el *autoencoder* la idea era reproducir la imagen de entrada luego de realizar una representación en el **espacio latente**:

$$
x \longrightarrow \hat{x}.
$$

Ahora queremos aprender:

$$
x \longrightarrow y,
$$

donde $y$ es la **clase correcta** de la imagen.

En este caso, la red no reconstruirá las imágenes. En cambio, producirá **10 logits**, uno por cada clase, y utilizaremos las etiquetas de Fashion-MNIST como objetivos.

> **Observación:** *train* se utiliza para actualizar parámetros, *validation* para comparar configuraciones y *test* se reserva para la evaluación final.

## Ejercicio 1) Preparar el entorno

### 1.1) Importar librerías

Usaremos las mismas herramientas básicas que en la guía anterior: PyTorch, `torchvision`, `DataLoader` y Matplotlib.

In [2]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

ModuleNotFoundError: No module named 'torchvision'

### 1.2) Reproducibilidad y dispositivo

Fijamos semillas y elegimos automáticamente GPU si está disponible.

## Ejercicio 2) Cargar y preparar Fashion-MNIST

### 2.1) Transformación

En clasificación no necesitamos que la salida de la red tenga el mismo rango que los píxeles de entrada. Por eso podemos reescalar las imágenes al intervalo `[-1,1]`.

Primero `ToTensor()` lleva los píxeles a `[0,1]`. Luego:

$$
x_{\mathrm{norm}}
=
\frac{x-0.5}{0.5}.
$$

Así obtenemos:

$$
x_{\mathrm{norm}}\in[-1,1].
$$

**Observación:** esta transformación **reescala** los píxeles, pero **no centra** los datos. Como la mayoría de los píxeles de Fashion-MNIST corresponden al fondo (negro, $x=0$, que se transforma en $x_{\mathrm{norm}}=-1$), la media de los píxeles normalizados es aproximadamente $-0.43$, y no cero. Lo importante aquí es que las entradas tomen valores de orden 1, de modo que las preactivaciones de la primera capa tengan una escala razonable.

Una alternativa es **estandarizar** las entradas utilizando la media y el desvío estándar de los píxeles de Fashion-MNIST ($\mu\approx0.286$, $\sigma\approx0.353$):

```python
transforms.Normalize((0.286,), (0.353,))
```

con lo cual las entradas quedan, en promedio, con media cero y varianza unitaria.

Esta elección es distinta de la del *autoencoder*: allí utilizábamos `[0,1]` porque la salida `Sigmoid` debía reconstruir los valores de los píxeles en `[0,1]`.

### 2.2) *Train*, *validation* y *test*

Usaremos la misma metodología que en la guía del *autoencoder*:

- 50.000 imágenes para *train*;
- 10.000 imágenes para *validation*;
- 10.000 imágenes oficiales para *test*.

No visualizaremos nuevamente el *dataset* (ya lo hicimos en la guía de práctico del *Autoencoder*). Nos concentraremos directamente en la **tarea de clasificación**.

### 2.3) Nombres de las clases

Fashion-MNIST codifica las categorías con **números enteros** de `0` a `9`.

El siguiente **diccionario** sólo se utilizará para traducir las **etiquetas numéricas** a **nombres legibles** cuando analicemos las predicciones.

## Ejercicio 3) Crear los `DataLoader`s

Como en el *autoencoder*, el entrenamiento se realiza **mini-batch por mini-batch**.

Usaremos *mini-batches* de 100 imágenes:

```text
images.shape = (B, 1, 28, 28)
labels.shape = (B,)
```

`train_loader` usa `shuffle=True`. *Validation* y *test* usan `shuffle=False`.

## Ejercicio 4) Comprender la tarea de clasificación

En un *autoencoder*:

$$
x\longrightarrow\hat{x}.
$$

En **clasificación supervisada**:

$$
x\longrightarrow y.
$$

Para una imagen, la red producirá un vector de 10 valores:

$$
\mathbb{z}
=
[z_0,z_1,\ldots,z_9]^T,
$$

donde cada $z_k$ es un *logit* para la clase $k$.

Los logits:

- son números reales;
- no son probabilidades;
- no tienen que estar entre 0 y 1;
- no tienen que sumar 1.

Conceptualmente, *softmax* transforma los *logits* en **probabilidades**:

$$
p_k
=
\frac{e^{z_k}}
{\sum_{j=0}^{9}e^{z_j}}.
$$

La clase predicha es la de mayor *logit*, equivalentemente la de **mayor probabilidad**:

$$
\hat{y}
=
\operatorname*{arg\,max}_k z_k.
$$

A diferencia del *autoencoder*, aquí **sí necesitamos las etiquetas** (por etiquetas nos referimos a los números enteros entre 0 y 9 que denotan cada clase, no nos referimos a los nombres de las clases) porque ellas indican cuál de las 10 clases es la correcta.

## Ejercicio 5) Definir la red de clasificación

Usaremos una MLP **completamente conectada**:

$$
784
\longrightarrow
128
\longrightarrow
64
\longrightarrow
10.
$$

Para un *mini-batch*:

$$
(B,1,28,28)
\longrightarrow
(B,784)
\longrightarrow
(B,128)
\longrightarrow
(B,64)
\longrightarrow
(B,10).
$$

La última capa produce 10 *logits*. No agregamos `Softmax` al modelo porque `CrossEntropyLoss` trabaja directamente con *logits*.

Incluiremos `Dropout(p=0.2)` después de las activaciones ocultas.

### 5.1) Comprobar dimensiones

Antes de entrenar, verificamos el *forward pass*.

Para $B$ imágenes y 10 clases esperamos:

$$
(B,1,28,28)
\longrightarrow
(B,10).
$$

## Ejercicio 6) Función de pérdida

Para **clasificación multiclase** utilizaremos la función de pérdida denominada **Entropía Cruzada** ($L$) (`CrossEntropyLoss`).

Para un ejemplo cuya clase verdadera es $y$, conceptualmente:

$$
L
=
-\log(p_y),
$$

donde $p_y$ es la probabilidad asignada por *softmax* a la **clase correcta**.

PyTorch implementa de manera numéricamente estable la combinación:

```text
logits → log-softmax → negative log-likelihood
```

Por eso:

- la red debe devolver **logits**;
- no aplicamos `Softmax` antes de la pérdida;
- las etiquetas son enteros entre `0` y `9`, no vectores *one-hot*.

## Ejercicio 7) Elegir el optimizador

Usaremos las mismas dos opciones que en la guía del *autoencoder*:

- SGD: `lr=0.01`;
- Adam: `lr=0.001`.

Comenzaremos con *Adam* y luego compararemos ambas configuraciones utilizando *validation*.

## Ejercicio 8) Implementar una época de entrenamiento

El `DataLoader` entrega *mini-batches*:

```python
for images, labels in data_loader:
```

Para cada *mini-batch*:

1. activar modo **entrenamiento**;
2. mover imágenes y etiquetas al dispositivo;
3. borrar gradientes anteriores;
4. hacer *forward*;
5. calcular *Cross-Entropy Loss*;
6. hacer *backpropagation*;
7. actualizar parámetros.

Como en el *Autoencoder*, `optimizer.step()` es la instrucción que modifica los pesos.

## Ejercicio 9) Implementar la evaluación

Durante evaluación:

- usamos `model.eval()`;
- desactivamos el cálculo de gradientes;
- no actualizamos parámetros;
- calculamos tanto pérdida (*loss*) como *accuracy*.

Para cada *mini-batch*:

```python
predictions = logits.argmax(dim=1)
```

selecciona la clase de mayor *logit* para cada imagen.

## Ejercicio 10) Crear el modelo y el optimizador

Ahora creamos una instancia nueva del modelo y asociamos el optimizador seleccionado a sus parámetros.

### 10.1) Predicciones antes del entrenamiento

Antes del entrenamiento, los pesos son aleatorios. Por eso no esperamos predicciones útiles.

Usaremos *validation*, NO *test*, para observar algunas predicciones iniciales.

Esta sección cumple el mismo rol conceptual que las reconstrucciones antes de  entrenar el *autoencoder*: mostrar que la arquitectura por sí sola no sabe cómo clasificar imágenes correctamente todavía.

## Ejercicio 11) Entrenar el modelo

En cada época:

```text
actualizar parámetros usando train
        ↓
evaluar train con los pesos finales
        ↓
evaluar validation con los mismos pesos
```

Así las métricas de *train* y *validation* corresponden exactamente al mismo estado del modelo.

Como en la guía del *autoencoder*, `train_one_epoch()` sólo actualiza pesos; las métricas se calculan después con `evaluate()`.

## Ejercicio 12) Graficar las métricas

En clasificación observaremos dos cantidades:

- `CrossEntropyLoss`;
- `accuracy`.

La **Pérdida** o **Error** (`CrossEntropyLoss` en este caso) es la función a partir de la cual se calculan los gradientes durante el proceso de entrenamiento de la red neuronal. La *accuracy* (exactitud) es una métrica interpretable de aciertos en el proceso de clasificación de las imágenes por parte de la red.

## Ejercicio 13) Comparar *SGD* y *Adam* utilizando *validation*

Hasta este punto no necesitamos utilizar *test*.

En los Ejercicios 10, 11 y 12 entrenamos el modelo paso a paso. Para comparar configuraciones conviene **encapsular todo el experimento en una función**:

```text
fijar semillas → crear modelo → crear optimizador → entrenar → registrar métricas
```

Esto tiene dos ventajas:

- cada corrida devuelve su propio `history`, de modo que los resultados de una configuración **no sobrescriben** los de otra;
- como las semillas se fijan **dentro** de la función, cada corrida es **reproducible**: la inicialización de los pesos, el orden de los *mini-batches* y las máscaras de *Dropout* quedan determinados por `seed`, independientemente de qué celdas se hayan ejecutado antes.

Mantener constantes:
- arquitectura;
- dropout;
- batch size;
- número de épocas.

Comparar principalmente:

- `valid_loss` y `valid_acc` **de la última época**;
- velocidad de convergencia;
- estabilidad de las curvas.

**Elegir la configuración utilizando sólo *validation*.**

### 13.1) Función para ejecutar un experimento completo

`make_optimizer()` crea el optimizador a partir de su nombre y `run_experiment()` ejecuta un entrenamiento completo, igual al de los Ejercicios 10 y 11, y devuelve el modelo entrenado y su `history`.

Para evaluar *train* usamos un `DataLoader` **sin** `shuffle`: el orden no afecta a la evaluación y así no consumimos números aleatorios del generador que determina el orden de los *mini-batches* de entrenamiento.

### 13.2) Entrenar ambas configuraciones

Entrenamos SGD y Adam con la **misma semilla** y los mismos hiperparámetros, salvo el optimizador y su *learning rate*. Guardamos cada resultado en el diccionario `results`.

### 13.3) Comparar las curvas

Graficamos ambas configuraciones juntas: línea continua para *validation* y línea discontinua para *train*.

### 13.4) (Opcional) Variabilidad entre semillas

Una única corrida por configuración no permite saber si una diferencia en *validation* es **real** o es producto del azar (inicialización, orden de los *mini-batches*, *Dropout*).

Repitiendo cada configuración con varias semillas podemos estimar la media y el desvío estándar de las métricas, y preguntarnos: **¿es la diferencia entre SGD y Adam mayor que la variabilidad entre corridas?**

> Esta celda entrena $2\times3=6$ modelos, por lo que puede demorar varios minutos.

### 13.5) Fijar la configuración seleccionada

Elegir el optimizador **a partir de los resultados sobre *validation*** y recuperar el modelo entrenado y su `history` correspondientes.

A partir de aquí, `model` es el **modelo final**: el que se evaluará sobre *test* en los ejercicios siguientes. Sus curvas de entrenamiento son las que corresponde mostrar en el informe.

## Ejercicio 14) Evaluar la configuración seleccionada sobre *test*

La metodología es la misma que en el *autoencoder*:

$$
\text{train}
\longrightarrow
\text{aprendizaje de parámetros},
$$

$$
\text{validation}
\longrightarrow
\text{comparación de configuraciones},
$$

$$
\text{test}
\longrightarrow
\text{evaluación final}.
$$

Recién ahora medimos el modelo sobre las 10.000 imágenes oficiales de *test*.

## Ejercicio 15) Visualizar errores de clasificación sobre *test*

Ahora sí podemos observar imágenes concretas del conjunto de *test* porque la configuración ya fue seleccionada.

Las imágenes **bien** clasificadas aportan poca información. Mostraremos en cambio imágenes **mal clasificadas**: observarlas ayuda a entender qué tipo de errores comete la red y si resultan razonables desde el punto de vista visual.

Como las imágenes están normalizadas a `[-1,1]`, para visualizarlas recuperamos el intervalo `[0,1]` invirtiendo la transformación: $x = 0.5\,x_{\mathrm{norm}} + 0.5$.

## Ejercicio 16) *Accuracy* por clase

La *accuracy* global puede ocultar diferencias entre categorías.

Para cada clase $k$ calculamos:

$$
\mathrm{accuracy}_k
=
\frac{\mathrm{correctos\ de\ la\ clase}\ k}
{\mathrm{ejemplos\ de\ la\ clase}\ k}.
$$

Esta evaluación se hace sobre el conjunto *test* **después de haber seleccionado el modelo**.

### 16.1) Matriz de confusión normalizada

La *accuracy* por clase resume cuántos ejemplos de cada clase fueron clasificados correctamente, pero no muestra **con qué otras clases se confunden los errores**.

Para obtener esa información utilizaremos una **matriz de confusión normalizada**.

Definimos una matriz $C$ de tamaño $10\times10$ donde:

- cada **fila** corresponde a la **clase verdadera**;
- cada **columna** corresponde a la **clase predicha**;
- $C_{ij}$ cuenta cuántos ejemplos cuya clase verdadera es $i$ fueron clasificados como clase $j$.

Por ejemplo:

$$
C_{6,0}
$$

indica cuántas imágenes cuya clase verdadera es `Shirt` fueron clasificadas como `T-shirt/top`.

La diagonal

$$
C_{00},C_{11},\ldots,C_{99}
$$

contiene las clasificaciones correctas.

Para facilitar la comparación entre clases, normalizaremos cada fila:

$$
C^{(\mathrm{norm})}_{ij}
=
\frac{C_{ij}}
{\sum_{k=0}^{9}C_{ik}}.
$$

Entonces cada fila suma aproximadamente 1:

$$
\sum_{j=0}^{9} C^{(\mathrm{norm})}_{ij}=1.
$$

De esta forma, cada elemento puede interpretarse como una **proporción de ejemplos de la clase verdadera $i$ que fueron asignados a la clase $j$**.

En una matriz de confusión normalizada ideal:

- la diagonal tendría valores cercanos a 1;
- los valores fuera de la diagonal serían cercanos a 0.

> **Importante:** esta matriz se calcula sobre `test_loader` porque ya hemos seleccionado la configuración final del modelo usando *validation*.

#### Visualización de la matriz

Representaremos la matriz normalizada como una imagen en escala de colores.

Para interpretar el gráfico:

- el eje vertical indica la **clase verdadera**;
- el eje horizontal indica la **clase predicha**;
- los valores de la diagonal representan **clasificaciones correctas**;
- los valores fuera de la diagonal muestran los pares de clases que la red confunde.

Como la matriz está normalizada por filas, los valores se encuentran entre 0 y 1.

#### Cómo interpretar el resultado

La diagonal de la matriz permite observar, clase por clase, qué proporción fue clasificada correctamente.

Por ejemplo, si para una determinada clase aparece:

$$
C^{(\mathrm{norm})}_{ii}=0.91,
$$

significa que aproximadamente el 91 % de los ejemplos de esa clase fueron clasificados correctamente.

Un valor fuera de la diagonal, por ejemplo:

$$
C^{(\mathrm{norm})}_{ij}=0.08,
$$

significa que aproximadamente el 8 % de los ejemplos cuya clase verdadera es $i$ fueron confundidos con la clase $j$.

La matriz de confusión complementa la *accuracy* global porque permite responder preguntas como:

- ¿qué clases son fáciles de reconocer?
- ¿qué clases son más difíciles?
- ¿qué pares de clases se confunden frecuentemente?
- ¿los errores tienen una estructura visual razonable?

En Fashion-MNIST es especialmente interesante observar posibles confusiones entre categorías visualmente semejantes, por ejemplo prendas de la parte superior del cuerpo.

## Ejercicio 17) Preguntas conceptuales

1. ¿Por qué en clasificación necesitamos las etiquetas y en el *autoencoder* no?
2. ¿Qué representa cada uno de los 10 *logits*?
3. ¿Por qué los *logits* no son probabilidades?
4. ¿Qué hace conceptualmente *softmax*?
5. ¿Por qué no agregamos `Softmax` al modelo cuando usamos `CrossEntropyLoss`?
6. ¿Qué recibe `CrossEntropyLoss` como predicción y como *target*?
7. ¿Qué hace `optimizer.zero_grad()`?
8. ¿Qué hace `loss.backward()`?
9. ¿Qué hace `optimizer.step()`?
10. ¿Por qué el bucle `for images, labels in data_loader` recorre *mini-batches*?
11. ¿Por qué evaluamos *train* nuevamente al terminar cada época?
12. ¿Qué diferencia existe entre `model.train()` y `model.eval()`?
13. ¿Qué ocurre con *Dropout* durante evaluación?
14. ¿Qué diferencia conceptual existe entre *loss* y *accuracy*?
15. ¿Por qué *validation* sirve para elegir una configuración y *test* no debería utilizarse para ello?
16. ¿Por qué *SGD* y *Adam* utilizan *learning rates* iniciales diferentes?

## Ejercicio 18) Experimentos sugeridos

Modificar una sola variable por vez ayuda a interpretar el efecto de cada **hiperparámetro**.

### 18.1) Learning rate con SGD

Probar:

```python
SGD_LR = 0.001
SGD_LR = 0.01
SGD_LR = 0.05
```

### 18.2) Dropout

Probar:

```python
DROPOUT_P = 0.0
DROPOUT_P = 0.2
DROPOUT_P = 0.5
```

### 18.3) Número de épocas

Probar:

```python
EPOCHS = 5
EPOCHS = 10
EPOCHS = 20
```

### 18.4) Optimizador

Comparar *SGD* y *Adam* con sus *learning rates* base.

### 18.5) Capacidad de aprendizaje del modelo

Cambiar, por ejemplo, la primera capa oculta:

```text
64 → 64 → 10
128 → 64 → 10
256 → 64 → 10
```

**Regla metodológica:** comparar configuraciones usando *validation* y reservar *test* para la configuración final.